In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Ensure value columns are numeric where present
for df, col in ((metric, 'value'), (trace, 'value'), (log, 'value')):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# --- metric.csv summary ---
metric_overall = pd.DataFrame([{
    'rows': len(metric),
    'min_timestamp_utc': metric['timestamp'].min(),
    'max_timestamp_utc': metric['timestamp'].max()
}])

metric_unique_cmdb = pd.Series(metric['cmdb_id'].unique()[:20], name='unique_cmdb_id').reset_index(drop=True)

# Group by (cmdb_id, kpi_name) and compute count and global P95 on the full series for each group
metric_groups = (metric
                 .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
                 .agg(count='size', p95=lambda x: x.quantile(0.95)))
# build group name and sort by count desc, limit to top 20
metric_top_groups = (metric_groups
                     .reset_index()
                     .assign(group_name=lambda df: df['cmdb_id'].astype(str) + '|' + df['kpi_name'].astype(str))
                     .sort_values('count', ascending=False)
                     .loc[:, ['group_name', 'count', 'p95']]
                     .head(20)
                     .reset_index(drop=True))

# --- trace.csv summary ---
trace_overall = pd.DataFrame([{
    'rows': len(trace),
    'min_timestamp_utc': trace['timestamp'].min(),
    'max_timestamp_utc': trace['timestamp'].max()
}])

trace_unique_cmdb = pd.Series(trace['cmdb_id'].unique()[:20], name='unique_cmdb_id').reset_index(drop=True)

trace_groups = (trace
                .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
                .agg(count='size', p95=lambda x: x.quantile(0.95)))
trace_top_groups = (trace_groups
                    .reset_index()
                    .assign(group_name=lambda df: df['cmdb_id'].astype(str) + '|' + df['trace_name'].astype(str))
                    .sort_values('count', ascending=False)
                    .loc[:, ['group_name', 'count', 'p95']]
                    .head(20)
                    .reset_index(drop=True))

# --- log.csv summary ---
log_overall = pd.DataFrame([{
    'rows': len(log),
    'min_timestamp_utc': log['timestamp'].min(),
    'max_timestamp_utc': log['timestamp'].max()
}])

log_unique_cmdb = pd.Series(log['cmdb_id'].unique()[:20], name='unique_cmdb_id').reset_index(drop=True)

log_groups = (log
              .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
              .agg(count='size', p95=lambda x: x.quantile(0.95)))
log_top_groups = (log_groups
                  .reset_index()
                  .assign(group_name=lambda df: df['cmdb_id'].astype(str) + '|' + df['log_name'].astype(str))
                  .sort_values('count', ascending=False)
                  .loc[:, ['group_name', 'count', 'p95']]
                  .head(20)
                  .reset_index(drop=True))

# --- error_logs.csv summary ---
error_overall = pd.DataFrame([{
    'rows': len(error_logs),
    'min_timestamp_utc': error_logs['timestamp'].min() if 'timestamp' in error_logs.columns else pd.NaT,
    'max_timestamp_utc': error_logs['timestamp'].max() if 'timestamp' in error_logs.columns else pd.NaT
}])

error_unique_cmdb = pd.Series(error_logs['cmdb_id'].unique()[:20], name='unique_cmdb_id').reset_index(drop=True)

# distinct (cmdb_id, message) samples up to 20
if 'message' in error_logs.columns:
    error_samples = (error_logs[['cmdb_id', 'message']]
                     .drop_duplicates()
                     .head(20)
                     .reset_index(drop=True))
else:
    error_samples = pd.DataFrame(columns=['cmdb_id', 'message'])

# Total count of error log rows
error_total_count = len(error_logs)

# Compact output variables
metric_overall, metric_unique_cmdb, metric_top_groups, \
trace_overall, trace_unique_cmdb, trace_top_groups, \
log_overall, log_unique_cmdb, log_top_groups, \
error_overall, error_unique_cmdb, error_samples, error_total_count

```
Out[1]:
```
```python
# Build a plain-English summary string from the previously computed summaries
metric_rows = int(metric_overall.loc[0, 'rows'])
metric_min = metric_overall.loc[0, 'min_timestamp_utc'].isoformat()
metric_max = metric_overall.loc[0, 'max_timestamp_utc'].isoformat()
metric_unique_list = metric_unique_cmdb.tolist()[:20]

trace_rows = int(trace_overall.loc[0, 'rows'])
trace_min = trace_overall.loc[0, 'min_timestamp_utc'].isoformat()
trace_max = trace_overall.loc[0, 'max_timestamp_utc'].isoformat()
trace_unique_list = trace_unique_cmdb.tolist()[:20]

log_rows = int(log_overall.loc[0, 'rows'])
log_min = log_overall.loc[0, 'min_timestamp_utc'].isoformat()
log_max = log_overall.loc[0, 'max_timestamp_utc'].isoformat()
log_unique_list = log_unique_cmdb.tolist()[:20]

error_rows = int(error_overall.loc[0, 'rows'])
error_total_count = int(error_total_count)

# Prepare top groups snippets (top 5 each) with concise fields
def top_groups_snippet(df, n=5):
    rows = []
    for _, r in df.head(n).iterrows():
        rows.append(f"{r['group_name']} (count={int(r['count'])}, p95={float(r['p95']):.6g})")
    return rows

metric_top_snip = top_groups_snippet(metric_top_groups, 5)
trace_top_snip = top_groups_snippet(trace_top_groups, 5)
log_top_snip = top_groups_snippet(log_top_groups, 5)

summary = (
    "Summary of loaded telemetry files (plain English):\n\n"
    "1) metric.csv\n"
    f"   - Total rows: {metric_rows}\n"
    f"   - Time range (UTC): {metric_min} to {metric_max}\n"
    f"   - Unique cmdb_id (up to 20): {', '.join(metric_unique_list)}\n"
    "   - Top 5 (cmdb_id|kpi_name) groups by row count with global P95:\n"
    + "\n".join([f"     - {s}" for s in metric_top_snip]) + "\n\n"
    "2) trace.csv\n"
    f"   - Total rows: {trace_rows}\n"
    f"   - Time range (UTC): {trace_min} to {trace_max}\n"
    f"   - Unique cmdb_id (up to 20): {', '.join(trace_unique_list)}\n"
    "   - Top 5 (cmdb_id|trace_name) groups by row count with global P95:\n"
    + "\n".join([f"     - {s}" for s in trace_top_snip]) + "\n\n"
    "3) log.csv\n"
    f"   - Total rows: {log_rows}\n"
    f"   - Time range (UTC): {log_min} to {log_max}\n"
    f"   - Unique cmdb_id (up to 20): {', '.join(log_unique_list)}\n"
    "   - Top 5 (cmdb_id|log_name) groups by row count with global P95:\n"
    + "\n".join([f"     - {s}" for s in log_top_snip]) + "\n\n"
    "4) error_logs.csv\n"
    f"   - Total rows: {error_rows}\n"
    f"   - Distinct (cmdb_id, message) samples returned: {len(error_samples)} (up to 20 shown earlier)\n"
    f"   - Total error log rows count: {error_total_count}\n\n"
    "Notes / highlights:\n"
    " - metric.csv contains 9,173 rows spanning ~24 minutes; many metric groups (top groups shown) have 25 samples each and notable P95 values (e.g., diskio and mem values are large for some services).\n"
    " - trace.csv contains 18,712 rows over the same time window; top trace groups include high P95s for several traces (examples shown).\n"
    " - log.csv contains 2,196 rows; some services have large log.row_count P95 (e.g., ts-basic-service ~2623.6, ts-auth-service ~354.4).\n"
    " - error_logs.csv had zero rows in this dataset (no error log samples returned).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   rows         min_timestamp_utc         max_timestamp_utc
0  9173 2024-01-23 15:40:00+00:00 2024-01-23 16:04:00+00:00, 0     ts-admin-basic-info-service
1          ts-admin-order-service
2          ts-admin-route-service
3         ts-admin-travel-service
4           ts-admin-user-service
5              ts-assurance-mongo
6            ts-assurance-service
7                   ts-auth-mongo
8                 ts-auth-service
9               ts-avatar-service
10               ts-basic-service
11              ts-cancel-service
12                ts-config-mongo
13              ts-config-service
14               ts-consign-mongo
15         ts-consign-price-mongo
16       ts-consign-price-service
17             ts-consign-service
18              ts-contacts-mongo
19            ts-contacts-service
Name: unique_cmdb_id, dtype: object,                                 group_name  count           p95
0                ts-voucher-service|socket     25  4.000000e+00
1          ts-admin-basic-info-service|cpu     25  8.628181e-01
2       ts-admin-basic-info-service|diskio     25  6.108838e+04
3   ts-admin-basic-info-service|latency-50     25  1.601298e-02
4   ts-admin-basic-info-service|latency-90     25  2.320260e-02
5          ts-admin-basic-info-service|mem     25  2.325943e+08
6       ts-admin-basic-info-service|socket     25  7.966667e+00
7     ts-admin-basic-info-service|workload     25  6.952200e-01
8               ts-admin-order-service|cpu     25  1.012270e-01
9            ts-admin-order-service|diskio     25  4.925993e+04
10              ts-admin-order-service|mem     25  2.004699e+08
11           ts-admin-order-service|socket     25  3.260000e+00
12              ts-admin-route-service|cpu     25  9.107118e-02
13           ts-admin-route-service|diskio     25  4.745741e+04
14              ts-admin-route-service|mem     25  1.919987e+08
15                    ts-travel2-mongo|mem     25  6.753698e+07
16                 ts-travel2-mongo|socket     25  6.000000e+00
17                  ts-travel2-service|cpu     25  4.992722e+00
18               ts-travel2-service|diskio     25  6.115888e+04
19           ts-travel2-service|latency-50     25  1.769590e-02,     rows         min_timestamp_utc         max_timestamp_utc
0  18712 2024-01-23 15:40:00+00:00 2024-01-23 16:04:00+00:00, 0                            root
1     ts-admin-basic-info-service
2         ts-admin-travel-service
3            ts-assurance-service
4                 ts-auth-service
5                ts-basic-service
6               ts-config-service
7        ts-consign-price-service
8              ts-consign-service
9             ts-contacts-service
10            ts-food-map-service
11                ts-food-service
12      ts-inside-payment-service
13         ts-order-other-service
14               ts-order-service
15             ts-payment-service
16      ts-preserve-other-service
17               ts-price-service
18               ts-route-service
19                ts-seat-service
Name: unique_cmdb_id, dtype: object,                                            group_name  count         p95
0   ts-user-service|trace.to_ts-user-service.row_c...     25   21.600000
1   root|trace.to_ts-admin-basic-info-service.dura...     25    0.010209
2   root|trace.to_ts-admin-basic-info-service.dura...     25    0.015466
3   root|trace.to_ts-admin-basic-info-service.erro...     25    0.000000
4   root|trace.to_ts-admin-basic-info-service.row_...     25   21.000000
5   root|trace.to_ts-admin-travel-service.duration...     25    0.130953
6   root|trace.to_ts-admin-travel-service.duration...     25    0.151701
7    root|trace.to_ts-admin-travel-service.error_rate     25    0.000000
8   ts-travel2-service|trace.to_ts-order-other-ser...     25    0.007604
9   ts-travel2-service|trace.to_ts-order-other-ser...     25    0.015645
10  ts-travel2-service|trace.to_ts-order-other-ser...     25    0.000000
11  ts-travel2-service|trace.to_ts-order-other-ser...     25   81.800000
12  ts-travel2-service|trace.to_ts-route-service.d...     25    0.003115
13  ts-travel2-service|trace.to_ts-route-service.d...     25    0.004007
14  ts-travel2-service|trace.to_ts-route-service.e...     25    0.000000
15  ts-travel2-service|trace.to_ts-route-service.r...     25  611.200000
16  ts-travel2-service|trace.from_ts-seat-service....     25    0.009863
17  ts-travel2-service|trace.from_ts-seat-service....     25    0.014474
18  ts-travel2-service|trace.from_ts-seat-service....     25    0.000000
19  ts-travel2-service|trace.from_ts-seat-service....     25  338.400000,    rows         min_timestamp_utc         max_timestamp_utc
0  2196 2024-01-23 15:40:00+00:00 2024-01-23 16:04:00+00:00, 0     ts-admin-basic-info-service
1         ts-admin-travel-service
2              ts-assurance-mongo
3            ts-assurance-service
4                   ts-auth-mongo
5                 ts-auth-service
6                ts-basic-service
7                 ts-config-mongo
8                ts-consign-mongo
9          ts-consign-price-mongo
10             ts-consign-service
11              ts-contacts-mongo
12            ts-contacts-service
13              ts-food-map-mongo
14            ts-food-map-service
15                  ts-food-mongo
16                ts-food-service
17        ts-inside-payment-mongo
18      ts-inside-payment-service
19         ts-order-other-service
Name: unique_cmdb_id, dtype: object,                                      group_name  count     p95
0   ts-admin-basic-info-service|log.error_count     25     0.0
1     ts-admin-basic-info-service|log.row_count     25    32.0
2       ts-admin-travel-service|log.error_count     25     0.0
3         ts-admin-travel-service|log.row_count     25    33.0
4            ts-assurance-service|log.row_count     25    15.6
5          ts-assurance-service|log.error_count     25     0.0
6                ts-basic-service|log.row_count     25  2623.6
7              ts-basic-service|log.error_count     25     0.0
8                 ts-auth-service|log.row_count     25   354.4
9               ts-auth-service|log.error_count     25     0.0
10            ts-contacts-service|log.row_count     25    63.6
11          ts-contacts-service|log.error_count     25     0.0
12           ts-consign-service|log.error_count     25     0.0
13             ts-consign-service|log.row_count     25    56.0
14          ts-security-service|log.error_count     25     0.0
15            ts-security-service|log.row_count     25    84.8
16               ts-order-service|log.row_count     25   439.0
17             ts-order-service|log.error_count     25     0.0
18         ts-order-other-service|log.row_count     25   492.6
19       ts-order-other-service|log.error_count     25     0.0,    rows min_timestamp_utc max_timestamp_utc
0     0               NaT               NaT, Series([], Name: unique_cmdb_id, dtype: object), Empty DataFrame
Columns: [cmdb_id, message]
Index: [], 0)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' if available, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# Ensure timestamp is timezone-aware UTC
if not pd.api.types.is_datetime64_any_dtype(metric['timestamp']):
    # if numeric unix seconds
    if pd.api.types.is_integer_dtype(metric['timestamp']) or pd.api.types.is_float_dtype(metric['timestamp']):
        metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
    else:
        metric['timestamp'] = pd.to_datetime(metric['timestamp'], utc=True)
else:
    # if datetime dtype but possibly tz-naive, ensure UTC tz
    if metric['timestamp'].dt.tz is None:
        metric['timestamp'] = metric['timestamp'].dt.tz_localize('UTC')

# Ensure value is numeric
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Candidate components
candidates = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service"
]

# Filter metric rows for candidate components
metric_candidates = metric[metric['cmdb_id'].isin(candidates)].copy()

# 1) For each (cmdb_id, kpi_name) group among these components compute global P95 and total row count
groups_full = (
    metric_candidates
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(count='size', global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# Prepare per-cmdb kpi_name list (up to 20 each)
kpi_list = (
    groups_full
    .groupby('cmdb_id')['kpi_name']
    .apply(lambda s: list(pd.Series(s.unique())[:20]))
    .reset_index()
    .rename(columns={'kpi_name': 'kpi_names_sample_up_to_20'})
)

# Note cmdb_ids that have no metric rows at all
cmdbs_with_rows = metric_candidates['cmdb_id'].unique().tolist()
cmdbs_no_rows = [c for c in candidates if c not in cmdbs_with_rows]

# 2) Filter metric rows to incident window (inclusive)
start = pd.to_datetime("2024-01-23 15:37:35", utc=True)
end = pd.to_datetime("2024-01-23 16:07:35", utc=True)
mask = (metric_candidates['timestamp'] >= start) & (metric_candidates['timestamp'] <= end)
metric_window = metric_candidates.loc[mask].copy()

# If metric_window is empty, prepare empty results accordingly
if metric_window.empty:
    final_table = pd.DataFrame(columns=[
        'group_name', 'cmdb_id', 'kpi_name', 'global_p95', 'count_in_window',
        'anomalies_count', 'earliest_anomaly_timestamp_utc', 'max_value_in_window'
    ])
else:
    # Merge global_p95 into window rows (global thresholds computed from full series)
    metric_window = metric_window.merge(
        groups_full[['cmdb_id', 'kpi_name', 'global_p95']],
        on=['cmdb_id', 'kpi_name'],
        how='left'
    )

    # Flag anomalies (value > global_p95)
    metric_window['is_anomaly'] = metric_window['value'] > metric_window['global_p95']

    # Window-level aggregates
    window_aggregates = (
        metric_window
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(
            count_in_window=('value', 'size'),
            max_value_in_window=('value', 'max'),
            anomalies_count=('is_anomaly', 'sum')
        )
        .reset_index()
    )

    # Earliest anomaly timestamp per group
    anomalies = metric_window[metric_window['is_anomaly']]
    if not anomalies.empty:
        earliest_anom = (
            anomalies
            .groupby(['cmdb_id', 'kpi_name'])['timestamp']
            .min()
            .reset_index()
            .rename(columns={'timestamp': 'earliest_anomaly_timestamp_utc'})
        )
    else:
        earliest_anom = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'earliest_anomaly_timestamp_utc'])

    # Combine aggregates with global p95 and full-series count
    final_table = (
        window_aggregates
        .merge(groups_full, on=['cmdb_id', 'kpi_name'], how='left')  # brings in count (full series) and global_p95
        .merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')
        .assign(
            group_name=lambda df: df['cmdb_id'].astype(str) + '|' + df['kpi_name'].astype(str)
        )
        .loc[:, [
            'group_name', 'cmdb_id', 'kpi_name', 'global_p95', 'count', 'count_in_window',
            'anomalies_count', 'earliest_anomaly_timestamp_utc', 'max_value_in_window'
        ]]
        .rename(columns={'count': 'count_full_series'})
    )

    # Format earliest_anomaly_timestamp_utc as UTC iso strings (keep NaT as-is)
    final_table['earliest_anomaly_timestamp_utc'] = final_table['earliest_anomaly_timestamp_utc'].dt.tz_convert('UTC')

# 3) Sort by anomalies_count desc, then by max_value_in_window desc, limit to top 20
if not final_table.empty:
    final_table = final_table.sort_values(
        by=['anomalies_count', 'max_value_in_window'],
        ascending=[False, False]
    ).reset_index(drop=True).head(20)

# Prepare compact per-candidate summary DataFrame showing kpi_name samples or note if no rows
summary_rows = []
for c in candidates:
    if c in cmdbs_no_rows:
        summary_rows.append({'cmdb_id': c, 'kpi_names_sample_up_to_20': np.nan, 'note': 'no metric rows found'})
    else:
        kp = kpi_list[kpi_list['cmdb_id'] == c]
        if not kp.empty:
            summary_rows.append({'cmdb_id': c, 'kpi_names_sample_up_to_20': kp.iloc[0]['kpi_names_sample_up_to_20'], 'note': ''})
        else:
            summary_rows.append({'cmdb_id': c, 'kpi_names_sample_up_to_20': np.nan, 'note': 'no kpi names found'})

kpi_summary = pd.DataFrame(summary_rows)

# Display compact results: final_table (top anomalies) and kpi_summary per candidate
final_table, kpi_summary

```
Out[2]:
```
```python
# Build a plain-English summary from the previously computed final_table and kpi_summary
lines = []
lines.append("Focused metric analysis for the five candidate components (plain English):")
lines.append("")
lines.append("Overview:")
lines.append("- For each (cmdb_id, kpi_name) we computed the global P95 using the full series (not filtered).")
lines.append("- Then we inspected the incident window 2024-01-23 15:37:35 UTC to 2024-01-23 16:07:35 UTC and counted in-window samples and anomalies (value > global P95).")
lines.append("")

if final_table.empty:
    lines.append("No metric rows found in the incident window for the target components.")
else:
    lines.append("Top groups with anomalies (sorted by anomalies_count then max value):")
    # Summarize up to top 20 rows already prepared in final_table
    for r in final_table.itertuples(index=False):
        grp = r.group_name
        gp95 = f"{r.global_p95:.6g}"
        full_ct = int(r.count_full_series)
        win_ct = int(r.count_in_window)
        anom_ct = int(r.anomalies_count)
        earliest = (r.earliest_anomaly_timestamp_utc.isoformat() 
                    if pd.notnull(r.earliest_anomaly_timestamp_utc) else "null")
        maxv = f"{r.max_value_in_window:.6g}"
        lines.append(f" - {grp}: global_p95={gp95}, full_series_count={full_ct}, in_window_count={win_ct}, anomalies={anom_ct}, earliest_anomaly={earliest}, max_in_window={maxv}")
    lines.append("")

# Per-component KPI name samples (compact)
lines.append("Per-component KPI names (samples up to 20 each):")
for r in kpi_summary.itertuples(index=False):
    cmdb = r.cmdb_id
    if isinstance(r.kpi_names_sample_up_to_20, float) and pd.isna(r.kpi_names_sample_up_to_20):
        lines.append(f" - {cmdb}: no metric rows found")
    else:
        # join kpi names
        kp_list = r.kpi_names_sample_up_to_20
        kp_str = ", ".join(str(x) for x in kp_list)
        lines.append(f" - {cmdb}: {kp_str}")

lines.append("")
lines.append("Key takeaways:")
lines.append("- All listed groups in the result set had 25 samples in the full series and 25 samples in the incident window.")
lines.append("- Each reported group shows 2 in-window anomalies (value > global P95).")
lines.append("- Memory (mem) and disk I/O (diskio) KPIs appear among the top anomalous groups across these services, along with cpu, workload, and socket for some services.")
lines.append("- Earliest anomaly timestamps for these groups are around 15:40–16:03 UTC (as shown per-group above).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    group_name            cmdb_id  kpi_name    global_p95  count_full_series  count_in_window  anomalies_count earliest_anomaly_timestamp_utc  max_value_in_window
0        ts-travel-service|mem  ts-travel-service       mem  2.749842e+08                 25               25                2      2024-01-23 16:03:00+00:00         2.749981e+08
1         ts-order-service|mem   ts-order-service       mem  2.668869e+08                 25               25                2      2024-01-23 16:03:00+00:00         2.669623e+08
2          ts-auth-service|mem    ts-auth-service       mem  2.639988e+08                 25               25                2      2024-01-23 16:03:00+00:00         2.640350e+08
3         ts-route-service|mem   ts-route-service       mem  2.580896e+08                 25               25                2      2024-01-23 16:01:00+00:00         2.580969e+08
4         ts-train-service|mem   ts-train-service       mem  2.524867e+08                 25               25                2      2024-01-23 16:03:00+00:00         2.525621e+08
5       ts-auth-service|diskio    ts-auth-service    diskio  1.042082e+05                 25               25                2      2024-01-23 15:40:00+00:00         1.661040e+05
6     ts-travel-service|diskio  ts-travel-service    diskio  5.853989e+04                 25               25                2      2024-01-23 15:50:00+00:00         6.813963e+04
7      ts-route-service|diskio   ts-route-service    diskio  6.150972e+04                 25               25                2      2024-01-23 15:40:00+00:00         6.566790e+04
8      ts-train-service|diskio   ts-train-service    diskio  5.901784e+04                 25               25                2      2024-01-23 15:42:00+00:00         5.981332e+04
9      ts-order-service|diskio   ts-order-service    diskio  5.538716e+04                 25               25                2      2024-01-23 15:52:00+00:00         5.661014e+04
10         ts-auth-service|cpu    ts-auth-service       cpu  3.897719e+01                 25               25                2      2024-01-23 16:00:00+00:00         3.967894e+01
11   ts-route-service|workload   ts-route-service  workload  3.526495e+01                 25               25                2      2024-01-23 15:40:00+00:00         3.549652e+01
12   ts-train-service|workload   ts-train-service  workload  2.161759e+01                 25               25                2      2024-01-23 15:50:00+00:00         2.235797e+01
13      ts-auth-service|socket    ts-auth-service    socket  1.653778e+01                 25               25                2      2024-01-23 15:55:00+00:00         1.758333e+01
14     ts-train-service|socket   ts-train-service    socket  1.696667e+01                 25               25                2      2024-01-23 15:59:00+00:00         1.700000e+01
15    ts-travel-service|socket  ts-travel-service    socket  1.576667e+01                 25               25                2      2024-01-23 15:58:00+00:00         1.685000e+01
16     ts-route-service|socket   ts-route-service    socket  1.390333e+01                 25               25                2      2024-01-23 15:53:00+00:00         1.398333e+01
17     ts-order-service|socket   ts-order-service    socket  1.288667e+01                 25               25                2      2024-01-23 15:47:00+00:00         1.343333e+01
18  ts-travel-service|workload  ts-travel-service  workload  1.053002e+01                 25               25                2      2024-01-23 15:51:00+00:00         1.149464e+01
19   ts-order-service|workload   ts-order-service  workload  6.992047e+00                 25               25                2      2024-01-23 16:00:00+00:00         7.564694e+00,              cmdb_id                          kpi_names_sample_up_to_20 note
0    ts-auth-service  [cpu, diskio, latency-50, latency-90, mem, soc...     
1   ts-order-service  [cpu, diskio, error, latency-50, latency-90, m...     
2   ts-route-service  [cpu, diskio, latency-50, latency-90, mem, soc...     
3   ts-train-service  [cpu, diskio, latency-50, latency-90, mem, soc...     
4  ts-travel-service  [cpu, diskio, error, latency-50, latency-90, m...     )```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import re

# Reuse existing 'trace' and 'candidates' if present
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

try:
    candidates
except NameError:
    candidates = [
        "ts-auth-service",
        "ts-order-service",
        "ts-route-service",
        "ts-train-service",
        "ts-travel-service"
    ]

# Ensure timestamp parsed as UTC datetimes (Unix seconds -> UTC)
if not pd.api.types.is_datetime64_any_dtype(trace['timestamp']):
    if pd.api.types.is_integer_dtype(trace['timestamp']) or pd.api.types.is_float_dtype(trace['timestamp']):
        trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
    else:
        trace['timestamp'] = pd.to_datetime(trace['timestamp'], utc=True)
else:
    if trace['timestamp'].dt.tz is None:
        trace['timestamp'] = trace['timestamp'].dt.tz_localize('UTC')

# Ensure value numeric
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, trace_name) across the full trace series
groups_full = (
    trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(count_full_series='size', global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# Build pattern to detect 'to_ts-<candidate>' or 'from_ts-<candidate>' in trace_name
# We'll check substring presence for each candidate for clarity
def matches_candidate_in_name(name):
    if pd.isna(name):
        return False
    for c in candidates:
        if f"to_ts-{c}" in name or f"from_ts-{c}" in name:
            return True
    return False

# Identify groups of interest: either cmdb_id is one of the candidates OR trace_name contains the to_/from_ pattern
groups_full['is_interest'] = groups_full['cmdb_id'].isin(candidates) | groups_full['trace_name'].apply(matches_candidate_in_name)

groups_of_interest = groups_full[groups_full['is_interest']].copy()

# 2) Filter trace rows to the incident window (inclusive)
start = pd.to_datetime("2024-01-23 15:37:35", utc=True)
end = pd.to_datetime("2024-01-23 16:07:35", utc=True)
mask_window = (trace['timestamp'] >= start) & (trace['timestamp'] <= end)
trace_window = trace.loc[mask_window].copy()

# Merge global_p95 info into window rows for groups of interest
# First limit to rows whose (cmdb_id, trace_name) exists in groups_of_interest
# Create a key to filter quickly
groups_key = set(zip(groups_of_interest['cmdb_id'], groups_of_interest['trace_name']))
trace_window['is_interest'] = trace_window.apply(lambda r: (r['cmdb_id'], r['trace_name']) in groups_key or matches_candidate_in_name(r['trace_name']) or (r['cmdb_id'] in candidates), axis=1)
trace_window_interest = trace_window[trace_window['is_interest']].copy()

# Merge global stats
trace_window_interest = trace_window_interest.merge(
    groups_of_interest[['cmdb_id', 'trace_name', 'count_full_series', 'global_p95']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# Flag anomalies in-window using the group's global_p95
trace_window_interest['is_anomaly'] = trace_window_interest['value'] > trace_window_interest['global_p95']

# Aggregate per (cmdb_id, trace_name) for those that have any rows in window
if not trace_window_interest.empty:
    window_agg = (
        trace_window_interest
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(
            count_in_window=('value', 'size'),
            anomalies_count=('is_anomaly', 'sum'),
            max_value_in_window=('value', 'max')
        )
        .reset_index()
    )

    # Earliest anomaly timestamp per group
    anomalies = trace_window_interest[trace_window_interest['is_anomaly']]
    if not anomalies.empty:
        earliest_anom = (
            anomalies
            .groupby(['cmdb_id', 'trace_name'])['timestamp']
            .min()
            .reset_index()
            .rename(columns={'timestamp': 'earliest_anomaly_timestamp_utc'})
        )
    else:
        earliest_anom = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'earliest_anomaly_timestamp_utc'])

    # Combine with full-series counts and global_p95
    trace_groups_window = (
        window_agg
        .merge(groups_of_interest[['cmdb_id', 'trace_name', 'count_full_series', 'global_p95']], on=['cmdb_id', 'trace_name'], how='left')
        .merge(earliest_anom, on=['cmdb_id', 'trace_name'], how='left')
        .assign(group_name=lambda df: df['cmdb_id'].astype(str) + '|' + df['trace_name'].astype(str))
        .loc[:, ['group_name', 'cmdb_id', 'trace_name', 'count_full_series', 'count_in_window', 'anomalies_count', 'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'global_p95']]
    )
else:
    # no rows in-window for interest groups
    trace_groups_window = pd.DataFrame(columns=['group_name', 'cmdb_id', 'trace_name', 'count_full_series', 'count_in_window', 'anomalies_count', 'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'global_p95'])

# Sort compactly as requested
if not trace_groups_window.empty:
    trace_groups_window = trace_groups_window.sort_values(by=['anomalies_count', 'max_value_in_window'], ascending=[False, False]).reset_index(drop=True).head(20)

# 3) Produce compact list of trace groups whose trace_name contains 'to_ts-<candidate>'
# Gather window rows where trace_name contains 'to_ts-<candidate>' (use any window rows or across whole file?)
# The instruction asks to show anomalies_count_in_window etc, so we focus on rows in the incident window.
def extract_callee(name):
    if pd.isna(name):
        return None
    for c in candidates:
        if f"to_ts-{c}" in name:
            return c
    return None

# Filter window interest rows for 'to_ts-<candidate>'
to_ts_rows = trace_window_interest.copy()
to_ts_rows['callee'] = to_ts_rows['trace_name'].apply(extract_callee)
to_ts_rows = to_ts_rows[~to_ts_rows['callee'].isna()].copy()

if not to_ts_rows.empty:
    # Aggregate per caller cmdb_id and trace_name (unique trace group)
    to_ts_agg = (
        to_ts_rows
        .groupby(['cmdb_id', 'trace_name', 'callee'], dropna=False)
        .agg(
            count_in_window=('value', 'size'),
            anomalies_count=('is_anomaly', 'sum'),
            max_value_in_window=('value', 'max')
        )
        .reset_index()
    )
    # Merge global_p95 from groups_of_interest
    to_ts_agg = to_ts_agg.merge(
        groups_of_interest[['cmdb_id', 'trace_name', 'global_p95']],
        on=['cmdb_id', 'trace_name'],
        how='left'
    )
    # Earliest anomaly timestamp
    anomalies_to = to_ts_rows[to_ts_rows['is_anomaly']]
    if not anomalies_to.empty:
        earliest_to = (
            anomalies_to
            .groupby(['cmdb_id', 'trace_name', 'callee'])['timestamp']
            .min()
            .reset_index()
            .rename(columns={'timestamp': 'earliest_anomaly_timestamp_utc'})
        )
        to_ts_agg = to_ts_agg.merge(earliest_to, on=['cmdb_id', 'trace_name', 'callee'], how='left')
    else:
        to_ts_agg['earliest_anomaly_timestamp_utc'] = pd.NaT

    to_ts_agg = to_ts_agg.rename(columns={'cmdb_id': 'caller_cmdb_id'})
    to_ts_agg = to_ts_agg.loc[:, ['caller_cmdb_id', 'callee', 'trace_name', 'global_p95', 'count_in_window', 'anomalies_count', 'earliest_anomaly_timestamp_utc', 'max_value_in_window']]

    # Sort and limit top 20
    to_ts_agg = to_ts_agg.sort_values(by=['anomalies_count', 'max_value_in_window'], ascending=[False, False]).reset_index(drop=True).head(20)
else:
    to_ts_agg = pd.DataFrame(columns=['caller_cmdb_id', 'callee', 'trace_name', 'global_p95', 'count_in_window', 'anomalies_count', 'earliest_anomaly_timestamp_utc', 'max_value_in_window'])

# Candidate-level note: indicate if any matching trace rows were found for each candidate (either as cmdb_id or referenced in trace_name)
candidate_notes = []
for c in candidates:
    # matched in groups_of_interest? (full-series)
    matched_groups = groups_of_interest[(groups_of_interest['cmdb_id'] == c) | (groups_of_interest['trace_name'].apply(lambda n: (not pd.isna(n)) and (f"to_ts-{c}" in n or f"from_ts-{c}" in n)))]
    matched_in_window = trace_window_interest[
        (trace_window_interest['cmdb_id'] == c) | (trace_window_interest['trace_name'].apply(lambda n: (not pd.isna(n)) and (f"to_ts-{c}" in n or f"from_ts-{c}" in n)))
    ]
    candidate_notes.append({
        'cmdb_id': c,
        'matched_groups_full_series_count': int(matched_groups.shape[0]),
        'matched_rows_in_window': int(matched_in_window.shape[0]),
        'note': '' if matched_groups.shape[0] > 0 else 'no matching trace groups found'
    })
candidate_summary = pd.DataFrame(candidate_notes)

# Compact outputs: groups_of_interest (summary), trace_groups_window (per-group in-window aggregates), to_ts_agg (caller->callee anomalies), candidate_summary
groups_of_interest_summary = groups_of_interest[['cmdb_id', 'trace_name', 'count_full_series', 'global_p95']].reset_index(drop=True).head(20)

groups_of_interest_summary, trace_groups_window, to_ts_agg, candidate_summary

```
Out[3]:
```
```python
# Build a plain-English summary string from previously computed trace analysis variables
n_groups = int(groups_of_interest_summary.shape[0]) if 'groups_of_interest_summary' in globals() else 0
total_groups = int(groups_of_interest.shape[0]) if 'groups_of_interest' in globals() else n_groups
n_window_groups = int(trace_groups_window.shape[0]) if 'trace_groups_window' in globals() else 0
n_to_ts = int(to_ts_agg.shape[0]) if 'to_ts_agg' in globals() else 0

# earliest anomaly across groups in-window
if n_window_groups > 0:
    earliest = trace_groups_window['earliest_anomaly_timestamp_utc'].min()
    earliest_str = earliest.isoformat()
else:
    earliest_str = "none"

# Prepare top 5 in-window example rows (compact)
examples = []
if n_window_groups > 0:
    for r in trace_groups_window.head(5).itertuples(index=False):
        examples.append(f"{r.group_name} (anomalies={int(r.anomalies_count)}, max={float(r.max_value_in_window):.6g}, p95={float(r.global_p95):.6g})")

# Candidate-level note summary
cand_notes_lines = []
if 'candidate_summary' in globals():
    for row in candidate_summary.itertuples(index=False):
        cand_notes_lines.append(f"{row.cmdb_id}: matched_groups={int(row.matched_groups_full_series_count)}, matched_rows_in_window={int(row.matched_rows_in_window)}")
else:
    cand_notes_lines.append("No candidate summary available.")

summary_lines = [
    "Trace-focused analysis for the five candidate services (plain English):",
    "",
    f"- Total distinct trace groups of interest (full series): {total_groups} (showing up to 20 in prior table).",
    f"- Trace groups with rows in the incident window (2024-01-23 15:37:35 to 2024-01-23 16:07:35 UTC): {n_window_groups}.",
    f"- Caller->callee ('to_ts-<candidate>') aggregated anomalies found in-window: {n_to_ts}.",
    "",
    f"- Earliest anomaly (any trace group in-window): {earliest_str}.",
    "",
    "Top example in-window trace groups (up to 5):"
] + [f"  - {e}" for e in examples] + [
    "",
    "Per-candidate quick counts:",
] + [f"  - {s}" for s in cand_notes_lines] + [
    "",
    "Key takeaways:",
    "- Many trace groups related to these services exist in the full series and were present in the incident window.",
    "- The in-window groups that show anomalies are primarily 'row_count' style trace metrics (large counts exceeding their global P95) across multiple services (examples above).",
    "- Each reported in-window group typically has 25 samples in the window and shows 2 anomalous samples exceeding the group's global P95 (consistent with earlier findings).",
    "- No caller->callee anomaly summary (from other services calling a candidate) was produced in the compact caller->callee table: either such cross-caller anomalies were not present in-window or the 'to_ts-<candidate>' anomalies were recorded under the callee's own cmdb_id rather than as separate caller metrics.",
    "",
    "Interpretation:",
    "- The traces indicate elevated row_count and related high-value trace metrics on the candidate services during the incident window (mid 15:40s to ~15:51s UTC).",
    "- This pattern suggests increased request/trace volume or bursty processing at the affected services rather than an isolated single caller producing the anomaly."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id                                         trace_name  count_full_series  global_p95
0    ts-auth-service                      trace.from_root.duration_mean                 25    0.924651
1    ts-auth-service                       trace.from_root.duration_p95                 25    1.732178
2    ts-auth-service                         trace.from_root.error_rate                 25    0.000000
3    ts-auth-service                          trace.from_root.row_count                 25   88.800000
4    ts-auth-service           trace.from_ts-auth-service.duration_mean                 25    0.010482
5    ts-auth-service            trace.from_ts-auth-service.duration_p95                 25    0.088277
6    ts-auth-service              trace.from_ts-auth-service.error_rate                 25    0.000000
7    ts-auth-service               trace.from_ts-auth-service.row_count                 25  354.400000
8    ts-auth-service             trace.to_ts-auth-service.duration_mean                 25    0.010482
9    ts-auth-service              trace.to_ts-auth-service.duration_p95                 25    0.088277
10   ts-auth-service                trace.to_ts-auth-service.error_rate                 25    0.000000
11   ts-auth-service                 trace.to_ts-auth-service.row_count                 25  354.400000
12  ts-order-service                      trace.from_root.duration_mean                 25    0.629017
13  ts-order-service                       trace.from_root.duration_p95                 25    0.688489
14  ts-order-service                         trace.from_root.error_rate                 25    0.000000
15  ts-order-service                          trace.from_root.row_count                 25   12.000000
16  ts-order-service  trace.from_ts-inside-payment-service.duration_...                 23    0.005213
17  ts-order-service  trace.from_ts-inside-payment-service.duration_p95                 23    0.009485
18  ts-order-service    trace.from_ts-inside-payment-service.error_rate                 23    0.000000
19  ts-order-service     trace.from_ts-inside-payment-service.row_count                 23   11.900000,                                            group_name            cmdb_id                                trace_name  count_full_series  count_in_window  anomalies_count earliest_anomaly_timestamp_utc  max_value_in_window  global_p95
0   ts-train-service|trace.from_ts-train-service.r...   ts-train-service     trace.from_ts-train-service.row_count                 25               25                2      2024-01-23 15:43:00+00:00               2744.0      2630.4
1   ts-train-service|trace.to_ts-train-service.row...   ts-train-service       trace.to_ts-train-service.row_count                 25               25                2      2024-01-23 15:43:00+00:00               2744.0      2630.4
2   ts-route-service|trace.from_ts-route-service.r...   ts-route-service     trace.from_ts-route-service.row_count                 25               25                2      2024-01-23 15:47:00+00:00               2184.0      2112.8
3   ts-route-service|trace.to_ts-route-service.row...   ts-route-service       trace.to_ts-route-service.row_count                 25               25                2      2024-01-23 15:47:00+00:00               2184.0      2112.8
4   ts-travel-service|trace.from_ts-travel-service...  ts-travel-service    trace.from_ts-travel-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00               1871.0      1753.4
5   ts-travel-service|trace.to_ts-travel-service.r...  ts-travel-service      trace.to_ts-travel-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00               1871.0      1753.4
6   ts-route-service|trace.from_ts-travel2-service...   ts-route-service   trace.from_ts-travel2-service.row_count                 25               25                2      2024-01-23 15:44:00+00:00                636.0       611.2
7   ts-order-service|trace.from_ts-order-service.r...   ts-order-service     trace.from_ts-order-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00                576.0       524.4
8   ts-order-service|trace.to_ts-order-service.row...   ts-order-service       trace.to_ts-order-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00                576.0       524.4
9   ts-travel-service|trace.to_ts-ticketinfo-servi...  ts-travel-service  trace.to_ts-ticketinfo-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00                392.0       374.0
10  ts-auth-service|trace.from_ts-auth-service.row...    ts-auth-service      trace.from_ts-auth-service.row_count                 25               25                2      2024-01-23 15:44:00+00:00                358.0       354.4
11  ts-auth-service|trace.to_ts-auth-service.row_c...    ts-auth-service        trace.to_ts-auth-service.row_count                 25               25                2      2024-01-23 15:44:00+00:00                358.0       354.4
12  ts-route-service|trace.from_ts-travel-service....   ts-route-service    trace.from_ts-travel-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00                343.0       325.6
13  ts-travel-service|trace.to_ts-route-service.ro...  ts-travel-service       trace.to_ts-route-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00                343.0       325.6
14  ts-travel-service|trace.from_ts-seat-service.r...  ts-travel-service      trace.from_ts-seat-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00                274.0       258.8
15  ts-train-service|trace.from_ts-travel-service....   ts-train-service    trace.from_ts-travel-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00                246.0       233.8
16  ts-travel-service|trace.to_ts-train-service.ro...  ts-travel-service       trace.to_ts-train-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00                246.0       233.8
17  ts-route-service|trace.from_ts-basic-service.r...   ts-route-service     trace.from_ts-basic-service.row_count                 25               25                2      2024-01-23 15:43:00+00:00                155.0       145.2
18  ts-train-service|trace.from_ts-basic-service.r...   ts-train-service     trace.from_ts-basic-service.row_count                 25               25                2      2024-01-23 15:43:00+00:00                155.0       145.2
19  ts-order-service|trace.from_ts-seat-service.ro...   ts-order-service      trace.from_ts-seat-service.row_count                 25               25                2      2024-01-23 15:51:00+00:00                137.0       129.4, Empty DataFrame
Columns: [caller_cmdb_id, callee, trace_name, global_p95, count_in_window, anomalies_count, earliest_anomaly_timestamp_utc, max_value_in_window]
Index: [],              cmdb_id  matched_groups_full_series_count  matched_rows_in_window note
0    ts-auth-service                                12                     300     
1   ts-order-service                                36                     884     
2   ts-route-service                                20                     500     
3   ts-train-service                                20                     500     
4  ts-travel-service                                48                    1192     )```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse metric and candidates if available; otherwise load metric
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

try:
    candidates
except NameError:
    candidates = [
        "ts-auth-service",
        "ts-order-service",
        "ts-route-service",
        "ts-train-service",
        "ts-travel-service"
    ]

# Ensure timestamp is datetime UTC
if not pd.api.types.is_datetime64_any_dtype(metric['timestamp']):
    if pd.api.types.is_integer_dtype(metric['timestamp']) or pd.api.types.is_float_dtype(metric['timestamp']):
        metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
    else:
        metric['timestamp'] = pd.to_datetime(metric['timestamp'], utc=True)
else:
    if metric['timestamp'].dt.tz is None:
        metric['timestamp'] = metric['timestamp'].dt.tz_localize('UTC')

# Ensure value numeric
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Target KPIs
target_kpis = ['mem', 'diskio']

# Filter metric rows to candidate components
metric_candidates = metric[metric['cmdb_id'].isin(candidates)].copy()

# 1) Compute global P95 and full-series count for each (cmdb_id, kpi_name) where kpi_name in target_kpis
groups_full = (
    metric_candidates[metric_candidates['kpi_name'].isin(target_kpis)]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(count_full_series='size', global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# Identify missing groups (candidate x kpi combos not present)
all_combinations = pd.DataFrame([(c, k) for c in candidates for k in target_kpis], columns=['cmdb_id', 'kpi_name'])
missing = all_combinations.merge(groups_full, on=['cmdb_id', 'kpi_name'], how='left', indicator=True)
missing = missing[missing['_merge'] == 'left_only'][['cmdb_id', 'kpi_name']]

# 2) Filter to incident window (inclusive)
start = pd.to_datetime("2024-01-23 15:37:35", utc=True)
end = pd.to_datetime("2024-01-23 16:07:35", utc=True)
mask = (metric_candidates['timestamp'] >= start) & (metric_candidates['timestamp'] <= end)
metric_window = metric_candidates.loc[mask].copy()

# Prepare results list
rows = []

# Process existing groups
for _, g in groups_full.iterrows():
    cmdb = g['cmdb_id']
    kpi = g['kpi_name']
    group_name = f"{cmdb}|{kpi}"
    global_p95 = float(g['global_p95'])
    count_full = int(g['count_full_series'])
    # window rows for this group
    w = metric_window[(metric_window['cmdb_id'] == cmdb) & (metric_window['kpi_name'] == kpi)].copy()
    count_in_window = int(len(w))
    if count_in_window > 0:
        # Determine anomalies (value > global_p95)
        w = w.sort_values('timestamp')
        w['is_anomaly'] = w['value'] > global_p95
        anomalies = w[w['is_anomaly']].copy()
        anomalies_count = int(len(anomalies))
        if anomalies_count > 0:
            earliest_anomaly_ts = anomalies['timestamp'].min()
            # list of anomaly timestamps and values limited to first 20
            anomalies_list = list(anomalies.sort_values('timestamp').head(20).apply(lambda r: (r['timestamp'].isoformat(), float(r['value'])), axis=1))
            # Check for consecutive anomaly run of length >= 2 minutes (adjacent anomaly timestamps differ by exactly 60s)
            # Convert timestamps to integer seconds
            ts_seconds = anomalies['timestamp'].astype('int64') // 10**9
            ts_sorted = np.sort(ts_seconds.values)
            if ts_sorted.size >= 2:
                diffs = np.diff(ts_sorted)
                has_consecutive_run = bool((diffs == 60).any())
            else:
                has_consecutive_run = False
        else:
            earliest_anomaly_ts = pd.NaT
            anomalies_list = []
            has_consecutive_run = False
        max_value_in_window = float(w['value'].max())
    else:
        anomalies_count = 0
        earliest_anomaly_ts = pd.NaT
        anomalies_list = []
        has_consecutive_run = False
        max_value_in_window = np.nan

    # Compute breach_ratio safely
    if np.isfinite(global_p95) and global_p95 != 0 and not np.isnan(max_value_in_window):
        breach_ratio = (max_value_in_window - global_p95) / global_p95
    elif np.isfinite(global_p95) and global_p95 == 0 and not np.isnan(max_value_in_window):
        # define as inf if global_p95 == 0 and max > 0
        breach_ratio = np.inf if max_value_in_window > 0 else 0.0
    else:
        breach_ratio = np.nan

    rows.append({
        'group_name': group_name,
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': global_p95,
        'count_full_series': count_full,
        'count_in_window': count_in_window,
        'anomalies_count': anomalies_count,
        'earliest_anomaly_timestamp_utc': earliest_anomaly_ts,
        'anomalies_list_first_up_to_20': anomalies_list,
        'max_value_in_window': max_value_in_window,
        'breach_ratio': float(breach_ratio) if np.isfinite(breach_ratio) else breach_ratio,
        'has_consecutive_anomaly_run_ge_2min': has_consecutive_run
    })

# Add explicit entries for missing groups
for _, m in missing.iterrows():
    cmdb = m['cmdb_id']
    kpi = m['kpi_name']
    rows.append({
        'group_name': f"{cmdb}|{kpi}",
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': np.nan,
        'count_full_series': 0,
        'count_in_window': 0,
        'anomalies_count': 0,
        'earliest_anomaly_timestamp_utc': pd.NaT,
        'anomalies_list_first_up_to_20': [],
        'max_value_in_window': np.nan,
        'breach_ratio': np.nan,
        'has_consecutive_anomaly_run_ge_2min': False
    })

result_df = pd.DataFrame(rows)

# Sort by breach_ratio desc (NaNs last), then earliest_anomaly_timestamp_utc asc
# For sorting, replace inf with large number; keep NaNs at end
result_df['breach_sort'] = result_df['breach_ratio'].replace([np.inf, -np.inf], np.finfo('float64').max)
result_df_sorted = result_df.sort_values(by=['breach_sort', 'earliest_anomaly_timestamp_utc'], ascending=[False, True]).drop(columns=['breach_sort']).reset_index(drop=True)

# Limit to top 20 rows
result_top = result_df_sorted.head(20)

# Small summary: group with largest breach_ratio and its earliest anomaly timestamp
if result_df['breach_ratio'].replace([np.inf, -np.inf], np.nan).dropna().size > 0:
    # choose max breach_ratio (treat inf as largest)
    # use original breach_ratio with inf preserved
    idx_max = result_df['breach_ratio'].replace([np.inf, -np.inf], np.finfo('float64').max).idxmax()
    largest_breach = result_df.loc[idx_max, ['group_name', 'breach_ratio', 'earliest_anomaly_timestamp_utc']]
else:
    largest_breach = pd.Series({'group_name': None, 'breach_ratio': np.nan, 'earliest_anomaly_timestamp_utc': pd.NaT})

# Keep outputs compact per rules
result_top_selected = result_top.loc[:, [
    'group_name', 'cmdb_id', 'kpi_name', 'global_p95', 'count_full_series',
    'count_in_window', 'anomalies_count', 'earliest_anomaly_timestamp_utc',
    'max_value_in_window', 'breach_ratio', 'has_consecutive_anomaly_run_ge_2min'
]]

# Display results (variables for IPython)
result_top_selected, result_top[['group_name','anomalies_list_first_up_to_20']].set_index('group_name'), largest_breach

```
Out[4]:
```
```python
# Build a plain-English summary from prior analysis variables and display it
lines = []
lines.append("Summary: mem and diskio analysis for the five candidate services (plain English).")
lines.append("")
lines.append("Key findings (groups sorted by breach ratio):")

# Use result_top_selected if available
try:
    df = result_top_selected
except NameError:
    df = None

if df is None or df.empty:
    lines.append("No mem/diskio groups found for the target services.")
else:
    for r in df.itertuples(index=False):
        grp = r.group_name
        gp95 = f"{r.global_p95:.6g}" if pd.notnull(r.global_p95) else "n/a"
        full_ct = int(r.count_full_series) if not pd.isnull(r.count_full_series) else 0
        win_ct = int(r.count_in_window) if not pd.isnull(r.count_in_window) else 0
        anom_ct = int(r.anomalies_count)
        earliest = r.earliest_anomaly_timestamp_utc.isoformat() if pd.notnull(r.earliest_anomaly_timestamp_utc) else "null"
        maxv = f"{r.max_value_in_window:.6g}" if pd.notnull(r.max_value_in_window) else "n/a"
        br = f"{r.breach_ratio:.6g}" if pd.notnull(r.breach_ratio) else "n/a"
        cons = bool(r.has_consecutive_anomaly_run_ge_2min)
        lines.append(f" - {grp}: global_p95={gp95}, full_count={full_ct}, in_window={win_ct}, anomalies={anom_ct}, earliest_anomaly={earliest}, max_in_window={maxv}, breach_ratio={br}, consecutive_run>=2min={cons}")

lines.append("")
# Largest breach summary
try:
    lb = largest_breach
    if pd.notnull(lb['group_name']):
        lb_grp = lb['group_name']
        lb_br = lb['breach_ratio']
        lb_ts = lb['earliest_anomaly_timestamp_utc'].isoformat() if pd.notnull(lb['earliest_anomaly_timestamp_utc']) else "null"
        lines.append(f"Largest breach: {lb_grp} with breach_ratio={lb_br:.6g}, earliest_anomaly={lb_ts}")
    else:
        lines.append("Largest breach: none found")
except Exception:
    lines.append("Largest breach: not available")

lines.append("")
lines.append("Short interpretation:")
lines.append("- Disk I/O for ts-auth-service shows the largest breach (≈59% above its global P95) with a consecutive anomaly run; this is the most significant metric deviation.")
lines.append("- Other services (ts-travel, ts-route, ts-order, ts-train) show diskio breaches as well but much smaller (breach_ratio 0.01–0.16).")
lines.append("- Memory (mem) spikes are present for all five services but the breach ratios are extremely small (~0.00003–0.0003), indicating minor mem increases relative to their historical P95.")
lines.append("- Each reported group had 25 samples in the full series and 25 samples in the incident window; anomalies_count was 2 for the reported groups.")
lines.append("")
lines.append("Conclusion:")
lines.append("- The primary actionable signal is elevated disk I/O on ts-auth-service (largest breach and consecutive anomaly run). Memory increases are present but negligible in relative magnitude.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 group_name            cmdb_id kpi_name    global_p95  count_full_series  count_in_window  anomalies_count earliest_anomaly_timestamp_utc  max_value_in_window  breach_ratio  has_consecutive_anomaly_run_ge_2min
0    ts-auth-service|diskio    ts-auth-service   diskio  1.042082e+05                 25               25                2      2024-01-23 15:40:00+00:00         1.661040e+05      0.593962                                 True
1  ts-travel-service|diskio  ts-travel-service   diskio  5.853989e+04                 25               25                2      2024-01-23 15:50:00+00:00         6.813963e+04      0.163986                                False
2   ts-route-service|diskio   ts-route-service   diskio  6.150972e+04                 25               25                2      2024-01-23 15:40:00+00:00         6.566790e+04      0.067602                                False
3   ts-order-service|diskio   ts-order-service   diskio  5.538716e+04                 25               25                2      2024-01-23 15:52:00+00:00         5.661014e+04      0.022081                                False
4   ts-train-service|diskio   ts-train-service   diskio  5.901784e+04                 25               25                2      2024-01-23 15:42:00+00:00         5.981332e+04      0.013479                                False
5      ts-train-service|mem   ts-train-service      mem  2.524867e+08                 25               25                2      2024-01-23 16:03:00+00:00         2.525621e+08      0.000299                                 True
6      ts-order-service|mem   ts-order-service      mem  2.668869e+08                 25               25                2      2024-01-23 16:03:00+00:00         2.669623e+08      0.000282                                 True
7       ts-auth-service|mem    ts-auth-service      mem  2.639988e+08                 25               25                2      2024-01-23 16:03:00+00:00         2.640350e+08      0.000137                                 True
8     ts-travel-service|mem  ts-travel-service      mem  2.749842e+08                 25               25                2      2024-01-23 16:03:00+00:00         2.749981e+08      0.000051                                 True
9      ts-route-service|mem   ts-route-service      mem  2.580896e+08                 25               25                2      2024-01-23 16:01:00+00:00         2.580969e+08      0.000028                                False,                                               anomalies_list_first_up_to_20
group_name                                                                 
ts-auth-service|diskio    [(2024-01-23T15:40:00+00:00, 166103.9791100711...
ts-travel-service|diskio  [(2024-01-23T15:50:00+00:00, 58792.26758721108...
ts-route-service|diskio   [(2024-01-23T15:40:00+00:00, 65667.89772778835...
ts-order-service|diskio   [(2024-01-23T15:52:00+00:00, 55759.83988275311...
ts-train-service|diskio   [(2024-01-23T15:42:00+00:00, 59813.31752880899...
ts-train-service|mem      [(2024-01-23T16:03:00+00:00, 252487884.8), (20...
ts-order-service|mem      [(2024-01-23T16:03:00+00:00, 266915566.9333333...
ts-auth-service|mem       [(2024-01-23T16:03:00+00:00, 264003310.9333333...
ts-travel-service|mem     [(2024-01-23T16:03:00+00:00, 274998135.4666666...
ts-route-service|mem      [(2024-01-23T16:01:00+00:00, 258096878.9333333..., group_name                           ts-auth-service|diskio
breach_ratio                                       0.593962
earliest_anomaly_timestamp_utc    2024-01-23 15:40:00+00:00
Name: 0, dtype: object)```
```